# Final Submission: LLM Experiment

## Goal

This notebook focuses on the final submission requirement to test at least one additional LLM and compare it against the Milestone 2 model under controlled conditions. The goal is to understand how model choice affects grounded RAG outputs when both models receive the same retrieved context and the same prompt template.

## Imports

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
import pandas as pd
from langchain_core.output_parsers import StrOutputParser

from src.rag_pipeline import (
    PROMPT_VARIANTS, 
    build_rag_chain, 
    build_context
  )
import pickle
from src.semantic import SemanticRetriever
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from src.preprocessing import find_repo_root

C:\Users\ruthy\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT = find_repo_root()

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray


### LLM Comparison

We compare two models with different parameter counts:

| Model | Parameters | Provider | Context window |
|-------|-----------|----------|----------------|
| `llama-3.1-8b-instant` | 8B (baseline) | Groq | 128k tokens |
| `llama-3.3-70b-versatile` | 70B (challenger) | Groq | 128k tokens |

Both models use the same retrieved context (top-5 documents from the semantic retriever)
and the same concise prompt template, the only variable is model size.

Five test queries span different difficulty levels: keyword-exact, vague intent, multi-condition,
product comparison and negative filter.

In [3]:
load_dotenv()

# Initialise both LLMs with identical settings except model name
llm_8b  = ChatGroq(model="llama-3.1-8b-instant",      temperature=0.0, max_tokens=512)
llm_70b = ChatGroq(model="llama-3.3-70b-versatile",   temperature=0.0, max_tokens=512)

print("Models initialised:")
print(f"  Baseline  : {llm_8b.model_name}")
print(f"  Challenger: {llm_70b.model_name}")

Models initialised:
  Baseline  : llama-3.1-8b-instant
  Challenger: llama-3.3-70b-versatile


In [4]:
# Load pre-processed data
data_path = PROJECT_ROOT / Path("data/processed/All_Beauty_clean.parquet")
df = pd.read_parquet(data_path)
documents = df.to_dict(orient="records")

In [5]:
# Load pre-built index from disk — no re-embedding of 701k documents
semantic_index_path = PROJECT_ROOT / "data" / "processed" / "semantic_index"

required_semantic_files = [
    semantic_index_path / "semantic_documents.pkl",
    semantic_index_path / "semantic_model_name.json",
    semantic_index_path / "semantic_embeddings.npy",
    semantic_index_path / "semantic_faiss.index",
]

# Load pre-built index from disk if available; otherwise build and save it.
if all(path.exists() for path in required_semantic_files):
    semantic = SemanticRetriever.load(semantic_index_path)
    print(f"Semantic retriever loaded from disk: {semantic_index_path}")

    # Also load raw documents for later use, e.g. quantitative evaluation
    with open(semantic_index_path / "semantic_documents.pkl", "rb") as f:
        documents = pickle.load(f)
else:
    print("Semantic artifacts not found. Building Semantic retriever...")
    semantic = SemanticRetriever(documents)
    semantic.save(semantic_index_path)
    print(f"Semantic retriever built and saved to: {semantic_index_path}")


def to_lc_docs(query: str, k: int = 5) -> list[Document]:
    """Convert semantic retriever results into LangChain Documents.

    Parameters
    ----------
    query : str
        User query.
    k : int, default=5
        Number of top retrieved documents.

    Returns
    -------
    list of langchain_core.documents.Document
        Retrieved documents in LangChain format.
    """
    return [
        Document(
            page_content=doc["text"],
            metadata={key: val for key, val in doc.items() if key != "text"},
        )
        for doc in semantic.search(query, top_k=k)
    ]


retriever = RunnableLambda(to_lc_docs)

print(
    f"Retriever ready — {len(semantic.documents):,} documents loaded, "
    "with no re-embedding when artifacts already exist."
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4007.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic retriever loaded from disk: c:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data\processed\semantic_index
Retriever ready — 701,092 documents loaded, with no re-embedding when artifacts already exist.


In [6]:
def dict_docs_to_lc_docs(docs: list[dict]) -> list[Document]:
    """Convert retrieval dicts into LangChain Documents.

    Parameters
    ----------
    docs : list of dict
        Retrieved document dictionaries.

    Returns
    -------
    list of langchain_core.documents.Document
        LangChain document objects.
    """
    return [
        Document(
            page_content=doc["text"],
            metadata={k: v for k, v in doc.items() if k != "text"},
        )
        for doc in docs
    ]


comparison_queries = [
    "What lip balm works best for extremely dry, chapped lips?",
    "something for sensitive skin that won't break me out",
    "fragrance-free moisturizer that also has SPF protection",
    "Is Burt's Bees or EOS better for dry lips?",
    "hair serum that doesn't make hair greasy or weigh it down",
]

PROMPT_NAME = "concise"
prompt = PROMPT_VARIANTS[PROMPT_NAME]
parser = StrOutputParser()

results_rows = []

for query in comparison_queries:
    # Retrieve ONCE with the semantic retriever
    docs_dict = semantic.search(query, top_k=5)
    docs_lc = dict_docs_to_lc_docs(docs_dict)
    context = build_context(docs_lc)

    chain_8b = prompt | llm_8b | parser
    chain_70b = prompt | llm_70b | parser

    answer_8b = chain_8b.invoke(
        {
            "context": context,
            "question": query,
        }
    )

    answer_70b = chain_70b.invoke(
        {
            "context": context,
            "question": query,
        }
    )

    results_rows.append(
        {
            "query": query,
            "prompt_variant": PROMPT_NAME,
            "n_docs": len(docs_dict),
            "context": context,
            "answer_8b": answer_8b,
            "answer_70b": answer_70b,
        }
    )

    print(f"Query: {query}")
    print(f"[8B]  {answer_8b[:250]}")
    print(f"[70B] {answer_70b[:250]}")
    print()

comparison_df = pd.DataFrame(results_rows)
comparison_df

Query: What lip balm works best for extremely dry, chapped lips?
[8B]  Based on the reviews, the following lip balms are recommended for extremely dry, chapped lips:

- Best of Many Lip Balms (ASIN: B06XJNDV5T) - Reviewer states it lasts the longest and is very effective.
- It's the best lip balm I've tried (ASIN: B07BY
[70B] According to reviews 1, 2, 3, 4, and 5, the respective lip balms work well for dry, chapped lips. Reviewers mention that these lip balms are effective in moisturizing and healing dry lips. However, the specific product that works best is not clear as

Query: something for sensitive skin that won't break me out
[8B]  Based on the reviews, the following products may be suitable for sensitive skin that won't break you out:

- B07ZQS3P1J (Rating: 5/5) - "Sensitive skin great for sensitive skin"
- B08RNQNFW1 (Rating: 5/5) - "Best Buy! suitable for sensitive skin."
- 
[70B] Based on the reviews, the following products may be suitable for sensitive skin and won't break y

,query,prompt_variant,n_docs,context,answer_8b,answer_70b
0,"What lip balm works best for extremely dry, ch...",concise,5,[1] ASIN: B06XJNDV5T | Product: Best of Many L...,"Based on the reviews, the following lip balms ...","According to reviews 1, 2, 3, 4, and 5, the re..."
1,something for sensitive skin that won't break ...,concise,5,[1] ASIN: B071YZ2J6S | Product: I have very se...,"Based on the reviews, the following products m...","Based on the reviews, the following products m..."
2,fragrance-free moisturizer that also has SPF p...,concise,5,[1] ASIN: B000SDI6OG | Product: Great | Rating...,"Based on the reviews, there is no product that...",Review [1] mentions a product that is fragranc...
3,Is Burt's Bees or EOS better for dry lips?,concise,5,[1] ASIN: B07CSD7N2B | Product: Just buy it!! ...,Neither Burt's Bees nor EOS is specifically me...,The reviews do not provide a direct comparison...
4,hair serum that doesn't make hair greasy or we...,concise,5,[1] ASIN: B000TFW8WE | Product: The best | Rat...,"Based on the reviews, the hair serum does not ...","Reviews [1], [2], [3], and [5] mention that th..."


In [7]:
DIVIDER = "=" * 70

for _, row in comparison_df.iterrows():
    print(DIVIDER)
    print(f"Query: {row['query']}")
    print()
    print("--- llama-3.1-8b-instant ---")
    print(row["answer_8b"])
    print()
    print("--- llama-3.3-70b-versatile ---")
    print(row["answer_70b"])
    print()

Query: What lip balm works best for extremely dry, chapped lips?

--- llama-3.1-8b-instant ---
Based on the reviews, the following lip balms are recommended for extremely dry, chapped lips:

- Best of Many Lip Balms (ASIN: B06XJNDV5T) - Reviewer states it lasts the longest and is very effective.
- It's the best lip balm I've tried (ASIN: B07BY1M4H4) - Reviewer states it heals and makes lips soft.
- Must have for dry lips (ASIN: B0011V3T8W) - Reviewer states it works better than any other lip balm they have used before.
- My must have balm (ASIN: B01LQXI3H6) - Reviewer states it is the most moisturizing lip balm they have ever tried.
- The only lip balm that works! (ASIN: B07KKN2G3X) - Reviewer states it is the only product that has really worked for their extremely dry lips.

All of these lip balms have received high ratings and positive reviews for their effectiveness on extremely dry, chapped lips.

--- llama-3.3-70b-versatile ---
According to reviews 1, 2, 3, 4, and 5, the respectiv

In [8]:
# Update the default LLM to the 70B challenger
llm = llm_70b
rag_chain = build_rag_chain(retriever, llm, prompt_variant="concise")
print(f"Default LLM updated to: {llm.model_name}")

Default LLM updated to: llama-3.3-70b-versatile


`llama-3.3-70b-versatile` was selected as the final default model. Compared with `llama-3.1-8b-instant`, it produced more careful, better-grounded answers and handled ambiguous, comparison-based, and multi-condition queries more effectively. The 8B model was faster and sometimes more direct, but it was also more likely to overstate conclusions not fully supported by the retrieved reviews.